# Multi-property comparison

Loads every YAML in `properties/`, runs the same scenario against each, ranks them by economic gain.

Drop new property YAMLs into the folder and re-run — no code changes needed.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from mortgage_calc import (
    breakeven_appreciation, breakeven_hold_period,
    load_all_properties, load_scenario, run_scenario,
)

SCENARIO = "../scenarios/base.yaml"

properties = load_all_properties("../properties")
s = load_scenario(SCENARIO)
buy, rent, ctx = s["buy"], s["rent"], s["ctx"]

print(f"Loaded {len(properties)} properties under scenario: {SCENARIO}\n")
for key, p in properties.items():
    print(f"  {key}: {p.name} (${p.price:,.0f})")

## Side-by-side comparison

In [ ]:
rows = []
for key, prop in properties.items():
    r = run_scenario(prop, buy, rent, ctx)
    be_app = breakeven_appreciation(prop, buy, rent, ctx)
    be_hold = breakeven_hold_period(prop, buy, rent, ctx)
    rows.append({
        "property": prop.name,
        "price": prop.price,
        "cash_in": r.cash_invested,
        "monthly_yr1": r.monthly_payment_year_1,
        "profit_on_sale": r.profit_on_sale,
        "cash_on_cash_pct": r.cash_on_cash_return_pct,
        "true_gain": r.true_economic_gain,
        "breakeven_app_pct": be_app * 100 if be_app else None,
        "breakeven_hold_yrs": be_hold,
    })

comparison = pd.DataFrame(rows).sort_values("true_gain", ascending=False).reset_index(drop=True)
comparison.round(0)

## Visualizing the ranking

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(4, len(properties) * 0.7)))
colors = ["#1D9E75" if g > 0 else "#D85A30" for g in comparison["true_gain"]]
ax.barh(comparison["property"], comparison["true_gain"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("True economic gain over hold period ($)")
ax.set_title(f"Property ranking — scenario: {SCENARIO.split('/')[-1]}")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Stress-test across scenarios

How do the rankings change if assumptions change?

In [ ]:
SCENARIOS = ["../scenarios/optimistic.yaml", "../scenarios/base.yaml", "../scenarios/pessimistic.yaml"]

stress_rows = []
for scenario_path in SCENARIOS:
    s = load_scenario(scenario_path)
    buy_s, rent_s, ctx_s = s["buy"], s["rent"], s["ctx"]
    for key, prop in properties.items():
        r = run_scenario(prop, buy_s, rent_s, ctx_s)
        stress_rows.append({
            "scenario": scenario_path.split("/")[-1].replace(".yaml", ""),
            "property": prop.name,
            "true_gain": r.true_economic_gain,
        })

stress_df = pd.DataFrame(stress_rows)
pivot = stress_df.pivot(index="property", columns="scenario", values="true_gain")
pivot = pivot[["pessimistic", "base", "optimistic"]]  # order columns
pivot.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, len(properties) * 0.6)))
pivot.plot(kind="barh", ax=ax, color=["#D85A30", "#888780", "#1D9E75"])
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("True economic gain ($)")
ax.set_title("How each property fares across optimism / base / pessimism")
plt.tight_layout()
plt.show()

## Interpreting

- **Robust buy:** wins in all 3 scenarios → strong candidate
- **Optimistic-only:** only wins in optimistic → risky, you're betting on appreciation
- **Always loses:** rent instead, or this property is a bad fit
- **Big spread between scenarios:** high variance — your assumptions matter a lot